# Full-data training
GPU required. This starts a NEW 30-epoch run. For an interrupted run use resume_training.ipynb instead. Dataset and weights are not included. See README.md.


In [ ]:
%pip -q install ultralytics==8.4.150
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import zipfile, shutil
p=Path('/content/drive/MyDrive/disaster_drone_backup/datasets/indoor_elder_v3_yolov8.zip')
assert p.is_file(), 'Place the licensed dataset ZIP at this private Drive path first'
shutil.copy2(p,'/content/dataset.zip')
with zipfile.ZipFile('/content/dataset.zip') as z:
    assert z.testzip() is None
    z.extractall('/content/disaster_source')


In [ ]:
# FULL DATA TRAINING — 30 epochs, persistent backups and final test
from pathlib import Path
import os,sys,json,shutil,hashlib,time,yaml
import torch
from ultralytics import YOLO
from IPython.display import display,HTML,Image,clear_output
assert torch.cuda.is_available(), 'GPU required'
assert Path('/content/drive/MyDrive').is_dir(), 'Mount Drive first'
source=Path('/content/disaster_source'); full=Path('/content/disaster_full_v2')
assert (source/'data.yaml').is_file(), 'Restore dataset ZIP first'
run_id=time.strftime('%Y%m%d_%H%M%S')
saved=Path('/content/drive/MyDrive/disaster_drone_backup/full_training')/run_id
saved.mkdir(parents=True,exist_ok=False)
status_display=display(HTML('<h1>전체 데이터 학습 준비 중</h1><p>라벨 변환·중복 검사 후 최대 30회 학습합니다.</p>'),display_id=True)
def update_status(title,detail):
    import html
    status_display.update(HTML('<section style="padding:24px;background:#102138;color:white;border-radius:12px"><h1>'+html.escape(title)+'</h1><p>'+html.escape(detail)+'</p><p>Drive: '+html.escape(str(saved))+'</p></section>'))
def digest(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''): h.update(b)
    return h.hexdigest()
def backup_file(src,dst):
    src,dst=Path(src),Path(dst); tmp=dst.with_name(dst.name+'.partial')
    shutil.copy2(src,tmp)
    assert digest(src)==digest(tmp),'Backup checksum mismatch'
    tmp.replace(dst)
classes=['person','fire','smoke','door','staircase']
cfg=yaml.safe_load((source/'data.yaml').read_text());names=cfg['names'];names=dict(enumerate(names)) if isinstance(names,list) else names
remap={int(i):classes.index('staircase' if n=='stairs' else n) for i,n in names.items() if n in classes or n=='stairs'}
assert len(remap)==5
# Pixel-exact duplicates: prioritize test, then validation, then training.
from PIL import Image as PILImage
seen=set();audit={};duplicates=0
full=full/run_id
for split in ['test','valid','train']:
    image_out=full/split/'images';label_out=full/split/'labels'
    image_out.mkdir(parents=True);label_out.mkdir(parents=True)
    count=0;objects=[0]*5
    for image in sorted((source/split/'images').iterdir()):
        if image.suffix.lower() not in ['.jpg','.jpeg','.png','.bmp','.webp']: continue
        label=source/split/'labels'/(image.stem+'.txt')
        assert label.is_file(),str(label)
        with PILImage.open(image) as im:
            im=im.convert('RGB');key=hashlib.sha256(str(im.size).encode()+im.tobytes()).digest()
        if key in seen: duplicates+=1;continue
        seen.add(key);rows=[]
        for row in label.read_text().splitlines():
            fields=row.split()
            if not fields: continue
            assert len(fields)==5,str(label)
            old=int(fields[0]);vals=list(map(float,fields[1:]))
            assert all(0<=v<=1 for v in vals) and vals[2]>0 and vals[3]>0,str(label)
            if old in remap:
                new=remap[old];rows.append(str(new)+' '+' '.join(fields[1:]));objects[new]+=1
        try: os.link(image,image_out/image.name)
        except OSError: shutil.copy2(image,image_out/image.name)
        (label_out/label.name).write_text('\n'.join(rows)+'\n' if rows else '')
        count+=1
    audit[split]={'images':count,'objects':dict(zip(classes,objects))}
    print('DATA READY',split,audit[split],flush=True)
(full/'data.yaml').write_text(yaml.safe_dump({'path':str(full),'train':'train/images','val':'valid/images','test':'test/images','names':classes}))
(saved/'audit.json').write_text(json.dumps({'splits':audit,'pixel_duplicates_removed':duplicates,'source':'https://universe.roboflow.com/zeka-s-workspace/indoor-elder-objection/dataset/3'},indent=2))
backup_file(full/'data.yaml',saved/'data.yaml')
# Start fresh: earlier quick-model validation selection used a different split audit.
model=YOLO('yolov8n.pt')
started=time.time()
def checkpoint(t):
    for item in [t.last,t.best,t.csv]:
        if Path(item).is_file(): backup_file(item,saved/Path(item).name)
    epoch=t.epoch+1;remaining=(time.time()-started)/epoch*(30-epoch)/60
    (saved/'status.json').write_text(json.dumps({'epoch':epoch,'max_epochs':30,'estimated_remaining_minutes':remaining,'complete':False}))
    update_status('전체 데이터 학습 '+str(epoch)+'/30회 완료','모델 Drive 검증 저장 완료 · 남은 학습 예상 '+str(round(remaining))+'분 (평가 시간 별도)')
model.add_callback('on_model_save',checkpoint)
update_status('전체 데이터 GPU 학습 중',str(audit['train']['images'])+'장 · 최대 30회 · Tesla T4')
model.train(data=str(full/'data.yaml'),epochs=30,imgsz=640,batch=16,device=0,workers=2,seed=42,patience=30,close_mosaic=5,save=True,plots=True,project='/content/full_training',name=run_id)
checkpoint(model.trainer)
update_status('전체 학습 완료 · 독립 테스트 평가 중','저장된 best.pt로 test 분할을 평가합니다.')
best=YOLO(str(saved/'best.pt'))
metrics=best.val(data=str(full/'data.yaml'),split='test',imgsz=640,batch=16,device=0,workers=2,plots=True,project='/content/full_evaluation',name=run_id)
report={'split':'test','GPU':torch.cuda.get_device_name(0),'metrics':{k:float(v) for k,v in metrics.results_dict.items()},'per_class':{},'speed_ms':metrics.speed,'audit':audit}
for i,c in enumerate(metrics.box.ap_class_index):
    p,r,ap50,ap=metrics.box.class_result(i)
    report['per_class'][best.names[int(c)]]={'precision':float(p),'recall':float(r),'AP50':float(ap50),'AP50_95':float(ap)}
(saved/'metrics.json').write_text(json.dumps(report,indent=2))
shutil.copytree(metrics.save_dir,saved/'test_evaluation',dirs_exist_ok=True)
import matplotlib.pyplot as plt
fig,axs=plt.subplots(1,2,figsize=(13,5),layout='constrained')
cn=list(report['per_class']);v=[report['per_class'][n]['AP50'] for n in cn]
b=axs[0].barh(cn,v);axs[0].set_xlim(0,1);axs[0].bar_label(b,fmt='%.3f');axs[0].set_title('Full training: TEST AP50')
import pandas as pd
hist=pd.read_csv(saved/'results.csv');hist.columns=hist.columns.str.strip();axs[1].plot(hist['epoch'],hist['metrics/mAP50(B)'],label='Validation mAP50');axs[1].legend();axs[1].set_xlabel('Epoch')
fig.savefig(saved/'summary_chart.png',dpi=160);plt.close(fig)
testimgs=sorted((full/'test/images').iterdir());chosen=[]
for c in range(5):
    candidates=[p for p in testimgs if any(int(r.split()[0])==c for r in (full/'test/labels'/(p.stem+'.txt')).read_text().splitlines() if r.strip())]
    chosen.extend(candidates[:2])
fig,axs=plt.subplots(2,5,figsize=(18,8),layout='constrained')
for ax,p in zip(axs.flat,chosen):
    pred=best.predict(str(p),device=0,conf=.25,verbose=False)[0];ax.imshow(pred.plot()[...,::-1]);ax.axis('off')
fig.suptitle('Full-data model predictions on held-out images');fig.savefig(saved/'predictions.jpg',dpi=150);plt.close(fig)
weak=sorted(report['per_class'],key=lambda n:report['per_class'][n]['AP50'])
notes='보완 우선 클래스: '+', '.join(weak[:2])+' · 계단 데이터 추가 수집 · 유사 장면 누수 점검 · 실제 D435i/Jetson 검증 · 탐지와 깊이 지도의 통합 실측 필요'
(saved/'improvements.txt').write_text(notes,encoding='utf-8')
(saved/'status.json').write_text(json.dumps({'complete':True,'epochs':len(hist)}))
update_status('전체 데이터 학습·테스트 완료','Test mAP50 '+format(metrics.box.map50,'.3f')+' · '+notes)
display(Image(filename=str(saved/'summary_chart.png')));display(Image(filename=str(saved/'predictions.jpg')))
print('FINAL SAVED',saved,flush=True)
